# Human Pose Estimation using MediaPipe

## Introduction
In this project, we utilize MediaPipe, a cross-platform framework for building multimodal applied machine learning pipelines, to perform human pose estimation. MediaPipe provides a robust and efficient solution for detecting and tracking human poses in real-time.

## Objectives
- To implement human pose estimation using MediaPipe.
- To analyze the accuracy and performance of the MediaPipe pose estimation model.
- To explore potential applications of human pose estimation in various fields such as sports, healthcare, and entertainment.

## Methodology
1. **Setup and Installation**
    - Install MediaPipe and other necessary libraries.
    - Set up the environment for running the pose estimation model.

2. **Data Collection**
    - Collect video data or use existing datasets for testing the pose estimation model.
    - Preprocess the data to ensure compatibility with the MediaPipe framework.

3. **Pose Estimation**
    - Implement the MediaPipe pose estimation model.
    - Run the model on the collected data to detect and track human poses.

4. **Analysis and Evaluation**
    - Evaluate the accuracy and performance of the pose estimation model.
    - Analyze the results and identify any limitations or areas for improvement.

5. **Applications**
    - Explore potential applications of human pose estimation in various fields.
    - Discuss how the results of this project can be applied in real-world scenarios.

## Implementation
### Setup and Installation


In [1]:
#Install Imports
import subprocess
import sys

# Function to install a package using pip
def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# List of required packages
required_packages = [
    "pandas",
    "numpy==1.25",
    "moviepy",
    "matplotlib",
    "seaborn",
    "basemap==1.4.1",
    "opencv-python",
    "mediapipe"
]

# Check and install each package
for package in required_packages:
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        install_package(package)

Installing numpy==1.25...



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip3.11 install --upgrade pip


Installing basemap==1.4.1...



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip3.11 install --upgrade pip


Installing opencv-python...



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip3.11 install --upgrade pip
2025-02-12 16:23:50.990306: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
# Import Required Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from moviepy import VideoFileClip
import cv2
import mediapipe as mp

# Set the aesthetic style of the plots
sns.set_style("whitegrid")

In [13]:
# DataFrame to store angles
angles_columns = ['frame', 'left_elbow_angle', 'left_knee_angle', 'left_shoulder_angle', 'left_hip_angle', 'left_ankle_angle',
            'right_elbow_angle', 'right_knee_angle', 'right_shoulder_angle', 'right_hip_angle', 'right_ankle_angle']

joints_columns = ['frame', 'left_elbow', 'left_knee', 'left_shoulder', 'left_hip', 'left_ankle', 'left_wrist','left_feet','right_shoulder',
                   'right_elbow', 'right_wrist', 'right_hip', 'right_knee', 'right_ankle','right_feet']


mp_joints_df = pd.DataFrame(columns=joints_columns)

mp_angles_df = pd.DataFrame(columns=angles_columns)

INMAXVFOLDER = '../videos/maxV/maxV/'

OUTMAXVFOLDER = '../FINAL/mediapipe_maxV/'

INACCELFOLDER = '../videos/accel/accel/'

OUTACCELFOLDER = '../FINAL/mediapipe_accel/'

In [5]:
def calculate_angle(a, b, c):
    a = np.array(a)  # First point
    b = np.array(b)  # Mid point
    c = np.array(c)  # End point
    
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    
    if angle > 180.0:
        angle = 360.0 - angle
    
    return angle

In [ ]:
# Initialize MediaPipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()
mp_drawing = mp.solutions.drawing_utils



# Function to process videos using MediaPipe Pose and calculate joint angles
def process_videos_with_mediapipe(formatted_side_path):
    
    # Iterate over files in the formatted side directory
    for filename in os.listdir(formatted_side_path):
        if filename.lower().endswith('.mov'):
            base_filename = filename.split('.')[0]
            # Get the list of all frame files in the folder
            frame_folder_path = os.path.join(formatted_side_path, base_filename)
            frame_files = [f for f in os.listdir(frame_folder_path) if f.startswith('frame_') and f.endswith('.jpg')]

            # Extract frame numbers from the filenames and sort them
            frame_numbers = sorted([int(f.split('_')[1].split('.')[0]) for f in frame_files])
            video_file_path = os.path.join(formatted_side_path, filename)
            process_single_video(video_file_path, frame_numbers, mp_angles_df, mp_joints_df)

def process_single_video(video_file_path, frame_numbers, mp_angles_df, mp_joints_df):
    cap = cv2.VideoCapture(video_file_path)
    
    if not cap.isOpened():
        print(f"Error opening video file {video_file_path}")
        return
    
    print(f"Processing video: {video_file_path}")

    frame_count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Convert the frame to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Process the frame with MediaPipe Pose
        results = pose.process(frame_rgb)

        left_shoulder = -1
        left_elbow = -1
        left_wrist = -1
        left_hip = -1
        left_knee = -1
        left_ankle = -1
        left_feet = -1
        
        right_shoulder = -1
        right_elbow = -1
        right_wrist = -1
        right_hip = -1
        right_knee = -1
        right_ankle = -1
        right_feet = -1

        #Calculate angles for left side
        left_elbow_angle = -1
        left_knee_angle = -1
        left_shoulder_angle = -1
        left_hip_angle = -1
        left_ankle_angle = -1
        
        # Calculate angles for right side
        right_elbow_angle = -1
        right_knee_angle = -1
        right_shoulder_angle = -1
        right_hip_angle = -1
        right_ankle_angle = -1
        
        # Draw pose landmarks on the frame
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
            
            # Extract landmarks
            landmarks = results.pose_landmarks.landmark

            # Get coordinates for left side
            left_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            left_elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
            left_wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]
            left_hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
            left_knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
            left_ankle = [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y]
            left_feet = [landmarks[mp_pose.PoseLandmark.LEFT_FOOT_INDEX.value].x, landmarks[mp_pose.PoseLandmark.LEFT_FOOT_INDEX.value].y]
            
            # Get coordinates for right side
            right_shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
            right_elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
            right_wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]
            right_hip = [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
            right_knee = [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y]
            right_ankle = [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]
            right_feet = [landmarks[mp_pose.PoseLandmark.RIGHT_FOOT_INDEX.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_FOOT_INDEX.value].y]

            # Get image dimensions
            image_height, image_width, _ = frame.shape
            
            # Convert normalized coordinates to pixel positions
            left_shoulder = [(left_shoulder[0] * image_width), (left_shoulder[1] * image_height)]
            left_elbow = [(left_elbow[0] * image_width), (left_elbow[1] * image_height)]
            left_wrist = [(left_wrist[0] * image_width), (left_wrist[1] * image_height)]
            left_hip = [(left_hip[0] * image_width), (left_hip[1] * image_height)]
            left_knee = [(left_knee[0] * image_width), (left_knee[1] * image_height)]
            left_ankle = [(left_ankle[0] * image_width), (left_ankle[1] * image_height)]
            left_feet = [(left_feet[0] * image_width), (left_feet[1] * image_height)]
            
            right_shoulder = [(right_shoulder[0] * image_width), (right_shoulder[1] * image_height)]
            right_elbow = [(right_elbow[0] * image_width), (right_elbow[1] * image_height)]
            right_wrist = [(right_wrist[0] * image_width), (right_wrist[1] * image_height)]
            right_hip = [(right_hip[0] * image_width), (right_hip[1] * image_height)]
            right_knee = [(right_knee[0] * image_width), (right_knee[1] * image_height)]
            right_ankle = [(right_ankle[0] * image_width), (right_ankle[1] * image_height)]
            right_feet = [(right_feet[0] * image_width), (right_feet[1] * image_height)]

            #Calculate angles for left side
            left_elbow_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
            left_knee_angle = calculate_angle(left_hip, left_knee, left_ankle)
            left_shoulder_angle = calculate_angle(left_hip, left_shoulder, left_elbow)
            left_hip_angle = calculate_angle(left_shoulder, left_hip, left_knee)
            left_ankle_angle = calculate_angle(left_knee, left_ankle, left_feet)  # Assuming vertical line for ankle
            
            # Calculate angles for right side
            right_elbow_angle = calculate_angle(right_shoulder, right_elbow, right_wrist)
            right_knee_angle = calculate_angle(right_hip, right_knee, right_ankle)
            right_shoulder_angle = calculate_angle(right_hip, right_shoulder, right_elbow)
            right_hip_angle = calculate_angle(right_shoulder, right_hip, right_knee)
            right_ankle_angle = calculate_angle(right_knee, right_ankle, right_feet)  # Assuming vertical line for ankle
   
                    # Annotate points/joints on the image
            for point in [left_shoulder, left_elbow, left_wrist, left_hip, left_knee, left_ankle, left_feet,
                        right_shoulder, right_elbow, right_wrist, right_hip, right_knee, right_ankle, right_feet]:
                cv2.circle(frame, tuple(np.multiply(point, [1, 1]).astype(int)), 5, (0, 0, 255), -1)
            # Annotate angles on the image
            cv2.putText(frame, f'Left Elbow: {int(left_elbow_angle)}', tuple(np.multiply(left_elbow, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Left Knee: {int(left_knee_angle)}', tuple(np.multiply(left_knee, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Left Shoulder: {int(left_shoulder_angle)}', tuple(np.multiply(left_shoulder, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Left Hip: {int(left_hip_angle)}', tuple(np.multiply(left_hip, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Left Ankle: {int(left_ankle_angle)}', tuple(np.multiply(left_ankle, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
            
            cv2.putText(frame, f'Right Elbow: {int(right_elbow_angle)}', tuple(np.multiply(right_elbow, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Right Knee: {int(right_knee_angle)}', tuple(np.multiply(right_knee, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Right Shoulder: {int(right_shoulder_angle)}', tuple(np.multiply(right_shoulder, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Right Hip: {int(right_hip_angle)}', tuple(np.multiply(right_hip, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'Right Ankle: {int(right_ankle_angle)}', tuple(np.multiply(right_ankle, [1, 1]).astype(int)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
            
            
        if frame_count in frame_numbers:
                # Get the name of the video
                videoname = video_file_path.split('/')[-1].split('.')[0]
                outdir = "../FINAL/mediapipe_"+videoname
                # Save the frame with annotated points
                output_frame_path = os.path.join(outdir, f"frame_{frame_count}_pose.jpg")
                cv2.imwrite(output_frame_path, frame)

                 # Store points into a DataFrame
                mp_joints_df = mp_joints_df.append({
                        'frame': f"frame_{frame_count}",
                        'left_shoulder': left_shoulder,
                        'left_elbow': left_elbow,
                        'left_wrist': left_wrist,
                        'left_hip': left_hip,
                        'left_knee': left_knee,
                        'left_ankle': left_ankle,
                        'left_feet': left_feet,
                        'right_shoulder': right_shoulder,
                        'right_elbow': right_elbow,
                        'right_wrist': right_wrist,
                        'right_hip': right_hip,
                        'right_knee': right_knee,
                        'right_ankle': right_ankle,
                        'right_feet': right_feet
                }, ignore_index=True)

                mp_angles_df = mp_angles_df.append({
                    'frame': f"frame_{frame_count}",
                    'left_elbow_angle': left_elbow_angle,
                    'left_knee_angle': left_knee_angle,
                    'left_shoulder_angle': left_shoulder_angle,
                    'left_hip_angle': left_hip_angle,
                    'left_ankle_angle': left_ankle_angle,
                    'right_elbow_angle': right_elbow_angle,
                    'right_knee_angle': right_knee_angle,
                    'right_shoulder_angle': right_shoulder_angle,
                    'right_hip_angle': right_hip_angle,
                    'right_ankle_angle': right_ankle_angle
                }, ignore_index=True)    
        
                mp_joints_df.to_json(os.path.join(outdir, 'keypoints.json'), orient='records')
                mp_angles_df.to_json(os.path.join(outdir, 'angles.json'), orient='records')
        # Display the frame
        cv2.imshow('Frame', frame)
        frame_count += 1
        # Press Q on keyboard to exit
        if cv2.waitKey(25) & 0xFF == ord('q'):
            break
    
    # Release the video capture object
    cap.release()
    cv2.destroyAllWindows()

    return mp_angles_df

# Process videos for each side using MediaPipe Pose
#for side in sides:
#    process_videos_with_mediapipe(side)

process_videos_with_mediapipe("../videos/")


I0000 00:00:1739421209.324933  305648 gl_context.cc:369] GL version: 2.1 (2.1 ATI-6.1.13), renderer: AMD Radeon Pro 5300M OpenGL Engine


Processing video: ../videos/maxV_close.MOV


W0000 00:00:1739421209.558923  902309 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1739421209.622509  902310 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_29891/1952055072.py:172: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  mp_joints_df = mp_joints_df.append({
/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_29891/1952055072.py:190: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  mp_angles_df = mp_angles_df.append({
/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_29891/1952055072.py:172: FutureWarning: The frame.append method i

Processing video: ../videos/accel.MOV


/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_29891/1952055072.py:172: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  mp_joints_df = mp_joints_df.append({
/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_29891/1952055072.py:190: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  mp_angles_df = mp_angles_df.append({
/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_29891/1952055072.py:172: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  mp_joints_df = mp_joints_df.append({
/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_29891/1952055072.py:190: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  mp_angles_df 

Processing video: ../videos/maxV_far.mov


/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_29891/1952055072.py:172: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  mp_joints_df = mp_joints_df.append({
/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_29891/1952055072.py:190: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  mp_angles_df = mp_angles_df.append({
/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_29891/1952055072.py:172: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  mp_joints_df = mp_joints_df.append({
/var/folders/j7/tpgdfswn2r31tk28rclvmx4r0000gn/T/ipykernel_29891/1952055072.py:190: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  mp_angles_df 

: 